# HighPitchCT — Stage 5 على Kaggle (`v0.8.0`): التعلّم العميق ثنائي المجال المشروط فيزيائياً (`60% ITF + 40% RDC`, `p ∈ {1.5, 2.0, 2.5}`)

1. فعّل **Internet = On** وأضف `GITHUB_TOKEN` في **Add-ons → Secrets** (ويمكنك تفعيل **GPU T4 x2** ليختار الدفتر تلقائيًا `--backend torch_cuda` صراحةً).
2. ينفّذ هذا الدفتر:
   - التحديث إلى `v0.8.0` وتشغيل الـ **`52` اختبارًا كاملةً** مع `-W error::DeprecationWarning`.
   - تشغيل `Stage 4` و`Stage 5` على المريض الحقيقي `C012` باستخدام البروتوكول الفيزيائي والسريري المعتمد:
     - **التوزيع الهجين للبيانات:** `60% ITF (Increased Table Feed, 64 rows)` + `40% RDC (Reduced Detector Coverage)`.
     - **المنهج السريري متعدد الخطوات:** `p ∈ {1.5, 2.0, 2.5}` مع التكيّف التلقائي لـ `native_pitch` المريض.
     - **الشبكة ثنائية المجال:** `SinoInpaintUNet (4-ch)` + `HardDataConsistencyLayer` + `DifferentiableParkerFBPBridge (ram_lak)` + `ImageRefineUNet (3-ch)`.
   - عرض جداول التقدم الكمي (`Prelim FBP -> Domain-1 SinoNet+DC -> Final Dual-Domain`) ولوحات المقارنة البصرية (`Visual QA PNGs`).


In [ ]:
import os
from pathlib import Path
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret('GITHUB_TOKEN')
repo = Path('/kaggle/working/HighPitchCT-Pipeline')
if not repo.exists():
    rc = os.system(f'git clone https://{token}@github.com/BrahimSoufghalem/HighPitchCT-Pipeline.git {repo}')
    assert rc == 0, 'فشل استنساخ المستودع؛ تحقق من Internet ومن GITHUB_TOKEN'
else:
    rc = os.system(f'git -C {repo} pull --ff-only')
    assert rc == 0, 'فشل تحديث المستودع عبر git pull'

%cd /kaggle/working/HighPitchCT-Pipeline
!python -m pip install -q -e .
!python -W error::DeprecationWarning -m unittest discover -s tests -t . -v


## 1. تشغيل `Stage 4` و`Stage 5` بالبروتوكول السريري (`60% ITF + 40% RDC` عبر `p ∈ {1.5, 2.0, 2.5}`)


In [ ]:
import torch

sim_backend = 'torch_cuda' if torch.cuda.is_available() else 'numpy'
dl_backend = 'torch_cuda' if torch.cuda.is_available() else 'torch'
print(f'Explicit backends -> sim: {sim_backend} | dl: {dl_backend}')

!hpct download --series-file configs/pilot_c012.csv --workers 4 --retries 5
!hpct stage4-sim --case-id C012 --scenario both --target-pitch 2.0 --pitches 1.5,2.0,2.5 --num-slices 1 --n-xy 256 --itf-slab-slices 11 --filter-name ram_lak --backend {sim_backend} --run-phantom-check --save-pairs-npz --workers 4
!hpct stage5-train --case-id C012 --scenario both --pitches 1.5,2.0,2.5 --itf-ratio 0.60 --num-slices 1 --image-size 256 --itf-slab-slices 7 --epochs 12 --lr 0.0025 --filter-name ram_lak --backend {dl_backend} --run-phantom-check --save-dataset-npz --save-model --workers 4


## 2. ملخص المقاييس الكمية لـ `Stage 5` (`Prelim FBP` $\rightarrow$ `Domain-1 SinoNet+DC` $\rightarrow$ `Final Dual-Domain`)


In [ ]:
import json
import pandas as pd

with open('/kaggle/working/highpitchct-data/manifests/stage5_dl_report.json', encoding='utf-8') as f:
    rep5 = json.load(f)

assert rep5['overall_pass'] and rep5['trained_series_count'] == 1, 'فشل تدقيق Stage 5 على C012!'

s5_phan = rep5['synthetic_phantom_dl_check']
s5_c012 = rep5['trained_series'][0]
summary_c012 = s5_c012['summary']

rows = []
for item in summary_c012['per_sample']:
    rows.append({
        'العينة (Sample)': item['sample_id'],
        'السيناريو': item['scenario'].upper(),
        'Pitch (Target / Eff / Native)': f"{item['target_pitch']:.1f} / {item['effective_pitch']:.2f} / {item['native_pitch']:.2f}",
        'نسبة الفجوة المفقودة': f"{100.0 * item['missing_fraction']:.1f}%",
        'RMSE Raw No-Fill [HU]': f"{item['raw_nofill_rmse_hu']:.2f}",
        'RMSE Prelim FBP [HU]': f"{item['prelim_rmse_hu']:.2f}",
        'RMSE Domain-1 (SinoNet+DC) [HU]': f"{item['domain1_dc_rmse_hu']:.2f}",
        'RMSE Final Dual-Domain [HU]': f"{item['final_dual_rmse_hu']:.2f}",
        'SSIM (Prelim -> Dual)': f"{item['prelim_ssim']:.4f} -> {item['final_dual_ssim']:.4f}",
        'Wedge MAE (Reproj -> SinoNet)': f"{item['reproj_missing_wedge_mae']:.4f} -> {item['sinonet_dc_missing_wedge_mae']:.4f}",
    })

display(pd.DataFrame(rows))


## 3. المقارنة البصرية الإلزامية لـ `Stage 5` (`Phantom` + `Real Patient C012 Dual-Domain Visual QA`)


In [ ]:
from IPython.display import Image, Markdown, display

figs = [
    ('Stage 5 — فانتوم الصدر ثلاثي الأبعاد: إثبات الجسر التفاضلي (< 0.05 HU) والتدريب الهجين 60% ITF + 40% RDC', s5_phan['visual_qa_png']),
    ('Stage 5 — المريض الحقيقي C012: التدرج البصري الكامل (Low-Pitch Target vs Prelim FBP vs Domain-1 SinoNet+DC vs Final Dual-Domain)', s5_c012['visual_qa_png']),
]

for title, png_path in figs:
    display(Markdown(f'### {title}'))
    display(Image(filename=png_path))
